# NUBASE2020 → `materials/library/nubase2020.csv`

The isotope registry (`materials.isotopes`) reads one table built here from the NUBASE2020 evaluation, one row
per nuclide or excited state: `Z, A, ZAID, symbol, mass, excitation, half_life, abundance`. Elements come
from the same table (the letters of each symbol), and the registry computes decay constants as ln2 / half-life.

**Source.** `raw/nubase_4.mas20.txt`, NUBASE2020: F.G. Kondev, M. Wang, W.J. Huang, S. Naimi, G. Audi,
Chin. Phys. C 45, 030001 (2021), from the AMDC (https://www-nds.iaea.org/amdc/).

Run the notebook from this directory; it overwrites `../materials/library/nubase2020.csv`.

In [1]:
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("raw/nubase_4.mas20.txt")
LIBRARY = Path("../materials/library")

## 1. The file format

NUBASE is a fixed-width text file; its header gives the columns (1-based, inclusive). We read every field by
its position, never by splitting on spaces: names such as `174Lum` have no space before the state letter.

In [2]:
print("".join(line for line in RAW.open() if line.startswith("#")))

# Nubase2020 citation: F.G. Kondev, M. Wang, W.J. Huang, S. Naimi, and G. Audi, Chin. Phys. C45, 030001 (2021)
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#                                     Data FORMAT 
#     column   quantity   format      description
#       1: 3   AAA           a3       Mass Number (AAA)
#       5: 8   ZZZi          a4       Atomic Number (ZZZ); i=0 (gs); i=1,2 (isomers); i=3,4 (levels); i=5 (resonance); i=8,9 (IAS)
#                                     i=3,4,5,6 can also indicate isomers (when more than two isomers are presented in a nuclide)
#     12: 16   A El          a5       A Element 
#     17: 17   s             a1       s=m,n (isomers); s=p,q (levels); s=r (reonance); s=i,j (IAS); 
#                                     s=p,q,r,x can also indicate isomers (when more than two isomers are presented in 

In [3]:
# (name, first column, last column), 1-based and inclusive as in the header above
COLUMNS = [
    ("A",           1,   3),   # mass number
    ("Z",           5,   7),   # atomic number
    ("state",       8,   8),   # i: 0 ground state, 1-9 excited states (isomers, levels, IAS, ...)
    ("name",       12,  16),   # mass number + element, e.g. '174Lu'
    ("letter",     17,  17),   # m, n (isomers), p, q, r, x (more isomers or levels), i, j (IAS)
    ("mass_excess", 19,  31),  # keV, '#' = from systematics
    ("excitation", 43,  54),   # keV, excited states only; 'non-exist' for states shown not to exist
    ("half_life",  70,  78),   # value, 'stbl' (stable) or 'p-unst' (particle unstable)
    ("unit",       79,  80),
    ("decays",    120, 209),   # decay modes; 'IS=' is the natural abundance in %
]


def fields(line: str) -> dict:
    """The raw text of each field of one data line."""
    return {name: line[first - 1:last].strip() for name, first, last in COLUMNS}


lines = [line.rstrip("\n") for line in RAW.open() if line.strip() and not line.startswith("#")]
raw = pd.DataFrame([fields(line) for line in lines])
print(len(raw), "data lines")
raw[raw.name.str.startswith("174Lu") | (raw.name == "24Na")]

5843 data lines


,A,Z,state,name,letter,mass_excess,excitation,half_life,unit,decays
197,024,011,0,24Na,,-8417.901,,14.9560,h,B-=100
198,024,011,1,24Na,m,-7945.694,472.2074,20.18,ms,IT~100;B-=0.05
199,024,011,8,24Na,i,-2450.53,5967.37,,,
3781,174,071,0,174Lu,,-55570.3,,3.31,y,B+=100
3782,174,071,1,174Lu,m,-55399.5,170.83,142,d,IT=99.38 2;EC=0.62 2
3783,174,071,2,174Lu,n,-55329.5,240.818,395,ns,IT=100
3784,174,071,3,174Lu,p,-55205.1,365.183,145,ns,IT=100
3785,174,071,4,174Lu,q,-53714.6,1855.7,194,ns,IT=100
3786,174,071,5,174Lu,r,-51501.9,4068.4,97,ns,IT=100
3787,174,071,6,174Lu,x,-49720.7,5849.6,242,ns,IT=100


## 2. Numbers

* **Mass.** Atomic mass in u from the mass excess: `A + Δ/931494.10242` (Δ in keV; 1 u = 931494.10242 keV).
  `#` marks values from systematics; they are kept. States NUBASE lists as `non-exist` have no mass and are dropped.
* **Excitation energy** [keV] of excited states, as NUBASE gives it (0 for ground states).
* **Half-life** [s]. Units are case-sensitive: `ys` is yoctoseconds, `My` megayears. NUBASE's year is
  365.2422 d. `stbl` → inf; `p-unst` and missing → NaN. Limits (`<`, `>`, `~`) keep their number.
* **Abundance** [%] from `IS=` in the decay field.

In [4]:
U_KEV = 931494.10242
YEAR = 365.2422 * 86400.0
UNITS = {"ys": 1e-24, "zs": 1e-21, "as": 1e-18, "fs": 1e-15, "ps": 1e-12, "ns": 1e-9, "us": 1e-6,
         "ms": 1e-3, "s": 1.0, "m": 60.0, "h": 3600.0, "d": 86400.0, "y": YEAR,
         "ky": 1e3 * YEAR, "My": 1e6 * YEAR, "Gy": 1e9 * YEAR, "Ty": 1e12 * YEAR, "Py": 1e15 * YEAR,
         "Ey": 1e18 * YEAR, "Zy": 1e21 * YEAR, "Yy": 1e24 * YEAR}


def number(text: str) -> float:
    """A NUBASE number: '#' (systematics) and limit signs (<, >, ~) dropped; NaN if none."""
    text = re.sub(r"[#<>~*]", "", text).strip()
    try:
        return float(text)
    except ValueError:
        return math.nan


def half_life(value: str, unit: str) -> float:
    """Half-life [s]: inf if stable, NaN if particle unstable or unknown."""
    if value.startswith("stbl"):
        return math.inf
    if value.startswith("p-unst") or not value:
        return math.nan
    if unit not in UNITS:
        raise ValueError(f"unknown half-life unit {unit!r}")
    return number(value) * UNITS[unit]


def abundance(decays: str) -> float:
    match = re.search(r"IS=([\d.]+)", decays)
    return float(match.group(1)) if match else math.nan


print(half_life("609.8", "s"), half_life("12.32", "y") / YEAR, half_life("stbl", ""), half_life("1", "ys"))

609.8 12.32 inf 1e-24


## 3. Names and ZAIDs

* `Z`, `A` and the state index come from their own columns; the element symbol from the name (`174Lu` → `Lu`).
* `symbol` = element + A + state letter: `Na24`, `Na24m`, `Lu174x`; the neutron is `n1`.
* `ZAID` = `1000 Z + A` for ground states and `10 (1000 Z + A) + i` for excited states, with `i` NUBASE's state
  index (column 8). The index is unique within a nuclide, unlike the letters (`x` can be state 4, 5 or 6).
* Caveat: an excited state's ZAID can equal the ground-state ZAID of a heavier element (B-11 state 8 →
  50118 = Sn-118), so states are looked up by `symbol`, which is unique.

In [5]:
rows = raw[raw.excitation != "non-exist"].copy()
print(len(raw) - len(rows), "states marked non-exist dropped:",
      ", ".join(raw.name[raw.excitation == "non-exist"] + raw.letter[raw.excitation == "non-exist"]))

rows["Z"] = rows.Z.astype(int)
rows["A"] = rows.A.astype(int)
rows["i"] = rows.state.astype(int)
element = rows.name.str.extract(r"^\d+([A-Za-z]+)$")[0]
rows["symbol"] = element + rows.A.astype(str) + rows.letter
rows["ZAID"] = np.where(rows.i == 0, 1000 * rows.Z + rows.A, 10 * (1000 * rows.Z + rows.A) + rows.i)

rows["mass"] = rows.A + rows.mass_excess.map(number) / U_KEV
rows["excitation"] = np.where(rows.i == 0, 0.0, rows.excitation.map(number))   # NaN if NUBASE gives none
rows["half_life"] = [half_life(v, u) for v, u in zip(rows.half_life, rows.unit)]
rows["abundance"] = rows.decays.map(abundance)

nubase = rows[["Z", "A", "ZAID", "symbol", "mass", "excitation", "half_life", "abundance"]]
nubase = nubase.sort_values(["Z", "A", "ZAID"]).reset_index(drop=True)
nubase[nubase.symbol.str.match(r"^(H1|Na24\w?|Co60\w?|Ag110\w?|Lu174\w?)$")]

16 states marked non-exist dropped: 76Cum, 84Gam, 84Asm, 85Nbn, 86Nbn, 117Lam, 138Pmm, 142Ndm, 144Csn, 152Pmn, 174Wm, 174Wn, 181Pbm, 190Tln, 196Pbm, 197Bin


,Z,A,ZAID,symbol,mass,excitation,half_life,abundance
1,1,1,1001,H1,1.007825,0.0000,inf,99.9855
207,11,24,11024,Na24,23.990963,0.0000,5.384160e+04,NaN
208,11,24,110241,Na24m,23.991470,472.2074,2.018000e-02,NaN
209,11,24,110248,Na24i,23.997369,5967.3700,NaN,NaN
828,27,60,27060,Co60,59.933816,0.0000,1.663492e+08,NaN
829,27,60,270601,Co60m,59.933878,58.5900,6.280200e+02,NaN
1953,47,110,47110,Ag110,109.906111,0.0000,2.456000e+01,NaN
1954,47,110,471101,Ag110m,109.906112,1.1120,6.600000e-07,NaN
1955,47,110,471102,Ag110n,109.906237,117.5900,2.158816e+07,NaN
3639,71,174,71174,Lu174,173.940343,0.0000,1.044534e+08,NaN


## 4. Checks

In [6]:
assert not nubase.symbol.duplicated().any(), nubase[nubase.symbol.duplicated(keep=False)]
ground = nubase[nubase.ZAID == 1000 * nubase.Z + nubase.A]
assert not ground.ZAID.duplicated().any()
assert nubase.mass.notna().all()
na24, na24m = (nubase.set_index("symbol").loc[s] for s in ("Na24", "Na24m"))
assert abs(na24.half_life / 3600 - 14.956) < 0.01                     # Na-24: 14.956 h
assert abs(na24m.excitation - 472.207) < 0.01                          # Na-24m: 472.2 keV
assert abs((na24m.mass - na24.mass) * U_KEV - na24m.excitation) < 0.01   # mass difference = excitation
assert nubase[nubase.abundance > 0].groupby("Z").abundance.sum().between(99.9, 100.1).all()   # natural abundances [%]
print(len(nubase), "states,", len(ground), "ground states")

5827 states, 3558 ground states


## 5. Write

In [7]:
nubase.to_csv(LIBRARY / "nubase2020.csv", index=False)
print("wrote", LIBRARY / "nubase2020.csv", len(nubase), "rows")

wrote ../materials/library/nubase2020.csv 5827 rows
